In [85]:
import json
import modelseedpy
import cobrakbase
from cobrakbase.kbaseapi_cache import KBaseCache
from modelseedpy import RastClient, MSBuilder

In [60]:
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache/')
rast = RastClient()

In [29]:
annotation = None
with open('./data/annotation.json', 'r') as fh:
    annotation = json.load(fh)

In [33]:
gene_to_annotation = {}
for role, genes in annotation.items():
    for g in genes:
        if g not in gene_to_annotation:
            gene_to_annotation[g] = set()
        gene_to_annotation[g].add(role)

In [16]:
pan_genome_range = {
    85: {},
    95: {},
}
with open('./data/core_data_comp_84_no_limit.json', 'r') as fh:
    pan_genome_range[85]['nolimit'] = json.load(fh)
with open('./data/core_data_comp_85.json', 'r') as fh:
    pan_genome_range[85]['limit'] = json.load(fh)
with open('./data/core_data_comp.json', 'r') as fh:
    pan_genome_range[95]['limit'] = json.load(fh)
with open('./data/core_data_comp_95_no_limit.json', 'r') as fh:
    pan_genome_range[95]['nolimit'] = json.load(fh)

In [7]:
set(pan_genome_range[85]['limit'].keys()) == set(pan_genome_range[85]['nolimit'].keys())

True

In [17]:
genome_id = 'Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.17.contigs__.RAST'
ani_cut = 95
for genome_id in pan_genome_range[ani_cut]['limit']:
    core = pan_genome_range[ani_cut]['limit'][genome_id]['core']
    core_nl = pan_genome_range[ani_cut]['nolimit'][genome_id]['core']
    if core is None:
        core = set()
    if core_nl is None:
        core_nl = set()
    print(genome_id, len(core), len(core_nl))

Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST 61 61
Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_concoct_out.33.contigs__.RAST 0 0
Salt_Pond_MetaGSF2_A_D2_MG_DASTool_bins_concoct_out.13.contigs__.RAST 0 0
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.26.contigs__.RAST 0 0
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST 1263 2021
Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_metabat.14.contigs__.RAST 44 44
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST 2 2
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.27.contigs__.RAST 3 3
Salt_Pond_MetaG_R2_B_D1_MG_DASTool_bins_concoct_out.40.contigs__.RAST 74 74
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.11.contigs__.RAST 0 0
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_metabat.12.contigs__.RAST 31 31
Salt_Pond_MetaGSF2_A_D2_MG_DASTool_bins_concoct_out.8.contigs__.RAST 0 0
Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_metabat.34.contigs__.RAST 0 0
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_metabat.9.c

In [76]:
#genome_id = 'Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.17.contigs__.RAST'
genome_id = 'Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST'
pan_data_nl = pan_genome_range[ani_cut]['nolimit'][genome_id]
core = pan_genome_range[ani_cut]['limit'][genome_id]['core']
core_nl = pan_data_nl['core']

if core is None:
    core = set()
if core_nl is None:
    core_nl = set()
print(genome_id, len(core), len(core_nl))

clusters_function = {}
for c_id in pan_data_nl['core']:
    f_set_count = {}
    for k in pan_data_nl['cluster_to_features'][c_id]:
        role_set = frozenset(gene_to_annotation.get(k, {'hypothetical protein'}))
        if role_set not in f_set_count:
            f_set_count[role_set] = 0
        f_set_count[role_set] += 1
    if len(f_set_count) == 1:
        clusters_function[c_id] = set(list(f_set_count)[0])
    else:
        ## SOLVE Conflicts
        #print('need solve!', c_id, len(f_set_count))
        pass

from modelseedpy.core.msgenome import normalize_role

genome = kbase.get_from_ws(genome_id, 155805)
nmz_role_to_feature_id = {}
for feature in genome.features:
    for role in feature.ontology_terms.get('RAST', []):
        nmz_role = normalize_role(role)
        if nmz_role not in nmz_role_to_feature_id:
            nmz_role_to_feature_id[nmz_role] = set()
        nmz_role_to_feature_id[nmz_role].add(feature.id)    

new_roles = {}
for c_id, roles in clusters_function.items():
    for role in roles:
        nmz_role = normalize_role(role)
        if nmz_role not in nmz_role_to_feature_id:
            if nmz_role not in new_roles:
                new_roles[nmz_role] = set()
            new_roles[nmz_role].add(role)

Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST 1263 2021


In [84]:
from modelseedpy.core.msgenome import MSFeature
feature_pangenome = MSFeature('pangenome', '')
for roles in new_roles.values():
    for role in roles:
        feature_pangenome.add_ontology_term('RAST', role)

In [28]:
pan_genome_range[ani_cut]['nolimit'][genome_id]['feature_id_to_genome']['QXYT01000146.1_5']

'54a56eb405bfd08feefe990b3e87073c2c39976ef73be61036658c1e46db842d'

In [86]:
template = kbase.get_from_ws('GramNegModelTemplateV5_test', 12218)

In [87]:
b = MSBuilder(genome, template, genome_id)

In [88]:
model_base = b.build(genome_id, annotate_with_rast=False)

In [89]:
model_base

Name,Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST
Memory address,7f330ea19310
Number of metabolites,1132
Number of reactions,1105
Number of genes,890
Number of groups,576
Objective expression,1.0*bio1 - 1.0*bio1_reverse_b18f7
Compartments,"Cytosol, Extracellular"


In [90]:
genome.add_features([feature_pangenome])

In [91]:
model_pan = MSBuilder(genome, template, genome_id).build(genome_id, annotate_with_rast=False)

In [92]:
model_pan

Name,Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST
Memory address,7f3307aae910
Number of metabolites,1185
Number of reactions,1233
Number of genes,891
Number of groups,650
Objective expression,1.0*bio1 - 1.0*bio1_reverse_b18f7
Compartments,"Cytosol, Extracellular"


In [97]:
for r in model_pan.reactions:
    if r.gene_reaction_rule == 'pangenome':
        #print(r.build_reaction_string(True))
        pass
    elif 'pangenome' in r.gene_reaction_rule:
        print()

UDP-N-acetylglucosamine [c0] + Undecaprenyl-diphospho-N-acetylmuramoyl-L-alanyl-D-glutamyl-meso-2-6-diaminopimeloyl-D-alanyl-D-alanine [c0] <=> UDP [c0] + Undecaprenyl-diphospho-N-acetylmuramoyl--N-acetylglucosamine-L-ala-D-glu-meso-2-6-diaminopimeloyl-D-ala-D-ala [c0]
ATP [c0] + dCDP [c0] <=> ADP [c0] + dCTP [c0]
H2O [c0] + 2 H+ [c0] + 3-Ureidopropanoate [c0] --> CO2 [c0] + NH3 [c0] + beta-Alanine [c0]
ATP [c0] + 2'-Deoxy-5-hydroxymethylcytidine-5'-diphosphate [c0] --> ADP [c0] + 2'-Deoxy-5-hydroxymethylcytidine-5'-triphosphate [c0]
NAD [c0] + CoA [c0] + 3-Oxopropanoate [c0] <=> NADH [c0] + CO2 [c0] + Acetyl-CoA [c0]
PPi [c0] + H+ [c0] + AICAR [c0] <=> PRPP [c0] + 5-Amino-4-imidazolecarboxyamide [c0]
CTP [c0] + 1,2-diisohexadecanoyl-sn-glycerol 3-phosphate [c0] --> PPi [c0] + H+ [c0] + CDP-1,2-diisohexadecanoylglycerol [c0]
NAD [c0] + S-(Hydroxymethyl)glutathione [c0] <=> NADH [c0] + H+ [c0] + S-Formylglutathione [c0]
NAD [c0] + CoA [c0] + 3-Oxo-2-methylpropanoate [c0] <=> NADH [c0] +

In [71]:
for k, v in new_roles.items():
    print(k)
    for vv in v:
        print('\t', vv)

uncharacterizedinnermembraneproteinrard
	 Uncharacterized inner membrane protein RarD
24dihydroxyhept2ene17dioicacidaldolaseec41252
	 2,4-dihydroxyhept-2-ene-1,7-dioic acid aldolase (EC 4.1.2.52)
ectoinehydrolase
	 Ectoine hydrolase
trnadihydrouridine2020asynthaseec13191
	 tRNA-dihydrouridine(20/20a) synthase (EC 1.3.1.91)
benzoatedegradationringcleavagehydrolase
	 benzoate degradation ring-cleavage hydrolase
sulfuroxidationcycletranscriptionalregulatorsoxrarsrfamily
	 Sulfur oxidation cycle transcriptional regulator SoxR, ArsR family
uroporphyrinogeniiimethyltransferaseec211107
	 Uroporphyrinogen-III methyltransferase (EC 2.1.1.107)
methylcrotonylcoacarboxylasecarboxyltransferasesubunitec6414
	 Methylcrotonyl-CoA carboxylase carboxyl transferase subunit (EC 6.4.1.4)
phosphatetransportsystemregulatoryproteinphou
	 Phosphate transport system regulatory protein PhoU
proteinmethioninesulfoxidereductasecatalyticsubunitmsrp
	 Protein-methionine-sulfoxide reductase catalytic subunit MsrP
pho

In [103]:
for cpx in template.reactions.rxn46184_c.complexes:
    for role in cpx.roles:
        print(role.name)

Formylmethanofuran dehydrogenase subunit C (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (molybdenum) subunit C (EC 1.2.99.5)
Formylmethanofuran dehydrogenase subunit A (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (tungsten) subunit C (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (tungsten) subunit D (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (molybdenum) subunit D (EC 1.2.99.5)
Formylmethanofuran dehydrogenase subunit B (EC 1.2.99.5)
Formylmethanofuran dehydrogenase subunit D (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (molybdenum) operon gene E
Formylmethanofuran dehydrogenase (molybdenum) subunit B (EC 1.2.99.5)
Formylmethanofuran dehydrogenase (tungsten) subunit B (EC 1.2.99.5)
